# FF 连接器连边分析

## 目标
- 解析 FF 站点的 Name 字段，确定来源高速和方向
- 枚举来源高速的所有站点，计算 OSRM 最短路径
- 找到最近的站点作为 FF 的上游连接
- 在地图上绘制路径和连接关系

In [1]:
import pandas as pd
import numpy as np
import re
import os
import requests
import folium
from tqdm import tqdm
import polyline

# ============== 配置 ==============

# 修正后的元数据文件
CORRECTED_META = "./output/d3_interpolation_correction/pems_d3_meta_corrected.csv"

# OSRM 服务器
OSRM_SERVER = "http://localhost:5000"

# 输出目录
OUTPUT_DIR = "./output/ff_connection_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")

配置完成！


## 1. 加载修正后的元数据

In [2]:
# 加载数据
meta_df = pd.read_csv(CORRECTED_META, dtype={'ID': str, 'Fwy': str})
print(f"总站点数: {len(meta_df)}")
print(f"\n类型分布:")
print(meta_df['Type'].value_counts())

总站点数: 1192

类型分布:
Type
ML    883
HV    281
FF     28
Name: count, dtype: int64


In [3]:
# 分离 FF 站点和其他站点
ff_stations = meta_df[meta_df['Type'] == 'FF'].copy()
ml_hv_stations = meta_df[meta_df['Type'].isin(['ML', 'HV'])].copy()

print(f"FF 站点数: {len(ff_stations)}")
print(f"ML/HV 站点数: {len(ml_hv_stations)}")

FF 站点数: 28
ML/HV 站点数: 1164


In [4]:
# 查看 FF 站点的 Name 字段
print("FF 站点 Name 示例:")
print(ff_stations[['ID', 'Fwy', 'Dir', 'Name', 'Abs_PM']].head(20).to_string(index=False))

FF 站点 Name 示例:
     ID Fwy Dir                Name  Abs_PM
 311930  50   E 5NB and 5SB to 50EB   3.788
 312569  99   N          99NB to 50 298.456
 314194  51   N        99NB to 50WB   0.081
 314803   5   S      FF 5SB -> 50EB 517.904
 314943  50   W         50WB to 5NB   3.655
 316301  50   E          80WB->50EB   0.576
 316364  50   E          50EB->51NB   5.496
 318258   5   S         80EB to 5NB 522.182
 318266   5   S         80WB to 5NB 522.183
 318276  50   W     99NB to 50WB FF   5.333
 318486   5   S       FF SB5 --> 50 518.400
 318533  50   E          50EB->99SB   5.284
 318587  51   N          EB RTE 244   8.243
 318588  51   S          EB RTE 244   8.225
 318736  80   E        50EB to 80EB  81.704
 318753  65   S          65SB->80WB  64.619
 318754  65   S          65SB->80EB  64.619
 318755  80   E          80EB->65NB 106.401
 318756  80   W          80WB->65NB 106.401
3026053   5   N         50EB to 5NB 517.915


## 2. 解析 FF Name 字段

### 关键逻辑
- **流入 FF**: FF 所在高速出现在 `to` 之后 → 搜索来源高速作为**上游**
- **流出 FF**: FF 所在高速出现在 `to` 之前 → 搜索目标高速作为**下游**

```
FF 在 50E 上，Name = "5NB to 50EB"
→ 50E 在 "to" 后面，这是流入
→ 搜索 5N 作为上游

FF 在 80W 上，Name = "80WB to 50WB"
→ 80W 在 "to" 前面，这是流出  
→ 搜索 50W 作为下游
```

In [5]:
def parse_ff_name(name, ff_fwy, ff_dir):
    """
    解析 FF 的 Name 字段，确定连接方向和目标高速
    
    返回:
        flow_type: 'inbound' (流入) 或 'outbound' (流出)
        connections: [(fwy, dir), ...] 需要搜索的高速列表
            - 流入: 搜索来源高速作为上游
            - 流出: 搜索目标高速作为下游
    """
    if pd.isna(name) or not name:
        return 'unknown', []
    
    name_upper = name.upper()
    
    # 方向映射
    dir_map = {
        'NB': 'N', 'SB': 'S', 'EB': 'E', 'WB': 'W',
        'NORTH': 'N', 'SOUTH': 'S', 'EAST': 'E', 'WEST': 'W',
        'N': 'N', 'S': 'S', 'E': 'E', 'W': 'W'
    }
    
    # 匹配模式: 数字 + 方向
    pattern = r'(?:I-|SR-|US-|CA-)?\s*(\d+)\s*-?\s*(NB|SB|EB|WB|N|S|E|W)'
    
    def extract_routes(text):
        """从文本中提取所有 (fwy, dir) 对"""
        routes = []
        for match in re.finditer(pattern, text, re.IGNORECASE):
            fwy = match.group(1)
            direction = dir_map.get(match.group(2).upper(), match.group(2).upper())
            routes.append((fwy, direction))
        return routes
    
    # 分割 "to" 前后
    if ' TO ' in name_upper:
        before_to, after_to = name_upper.split(' TO ', 1)
        routes_before = extract_routes(before_to)
        routes_after = extract_routes(after_to)
        
        # 检查 FF 所在高速在哪边
        ff_in_before = any(r[0] == ff_fwy and r[1] == ff_dir for r in routes_before)
        ff_in_after = any(r[0] == ff_fwy and r[1] == ff_dir for r in routes_after)
        
        if ff_in_after and not ff_in_before:
            # FF 高速在 "to" 后面 → 流入，搜索前面的作为上游
            return 'inbound', routes_before
        elif ff_in_before and not ff_in_after:
            # FF 高速在 "to" 前面 → 流出，搜索后面的作为下游
            return 'outbound', routes_after
        else:
            # 无法确定，返回所有非本高速的路线
            all_routes = routes_before + routes_after
            other_routes = [(f, d) for f, d in all_routes if f != ff_fwy]
            return 'unknown', other_routes
    else:
        # 没有 "to"，尝试提取所有非本高速的路线
        all_routes = extract_routes(name_upper)
        other_routes = [(f, d) for f, d in all_routes if f != ff_fwy]
        return 'unknown', other_routes


# 测试解析
test_cases = [
    # (name, ff_fwy, ff_dir)
    ("5NB and 5SB to 50EB", '50', 'E'),      # 流入 50E，上游是 5N/5S
    ("80WB to 50WB", '80', 'W'),              # 流出 80W，下游是 50W
    ("80WB to 50WB", '50', 'W'),              # 流入 50W，上游是 80W
    ("I-80 WB to SR-99 NB", '80', 'W'),       # 流出 80W，下游是 99N
    ("99 NB to 50 WB", '99', 'N'),            # 流出 99N，下游是 50W
    ("Jct 80", '50', 'E'),                    # 无法确定
]

print("解析测试:")
for name, ff_fwy, ff_dir in test_cases:
    flow_type, connections = parse_ff_name(name, ff_fwy, ff_dir)
    print(f"  FF在{ff_fwy}{ff_dir}, Name='{name}'")
    print(f"    → {flow_type}, 连接: {connections}")

解析测试:
  FF在50E, Name='5NB and 5SB to 50EB'
    → inbound, 连接: [('5', 'N'), ('5', 'S')]
  FF在80W, Name='80WB to 50WB'
    → outbound, 连接: [('50', 'W')]
  FF在50W, Name='80WB to 50WB'
    → inbound, 连接: [('80', 'W')]
  FF在80W, Name='I-80 WB to SR-99 NB'
    → outbound, 连接: [('99', 'N')]
  FF在99N, Name='99 NB to 50 WB'
    → outbound, 连接: [('50', 'W')]
  FF在50E, Name='Jct 80'
    → unknown, 连接: []


In [6]:
# 解析所有 FF 站点
ff_parsed = []

for _, row in ff_stations.iterrows():
    flow_type, connections = parse_ff_name(row['Name'], row['Fwy'], row['Dir'])
    
    ff_parsed.append({
        'ID': row['ID'],
        'Fwy': row['Fwy'],
        'Dir': row['Dir'],
        'Abs_PM': row['Abs_PM'],
        'Latitude': row['Latitude'],
        'Longitude': row['Longitude'],
        'Name': row['Name'],
        'Flow_Type': flow_type,
        'Connections': connections,
        'Num_Connections': len(connections)
    })

ff_parsed_df = pd.DataFrame(ff_parsed)

print(f"解析完成: {len(ff_parsed_df)} 个 FF")
print(f"\n流向类型分布:")
print(ff_parsed_df['Flow_Type'].value_counts())
print(f"\n连接数量分布:")
print(ff_parsed_df['Num_Connections'].value_counts().sort_index())

解析完成: 28 个 FF

流向类型分布:
Flow_Type
unknown     15
inbound      7
outbound     6
Name: count, dtype: int64

连接数量分布:
Num_Connections
0     4
1    22
2     2
Name: count, dtype: int64


In [7]:
# 显示各类型的 FF 示例
print("=== 流入 FF (inbound) ===")
inbound = ff_parsed_df[ff_parsed_df['Flow_Type'] == 'inbound']
for _, row in inbound.head(5).iterrows():
    print(f"  {row['ID']} 在 {row['Fwy']}{row['Dir']}: {row['Name']}")
    print(f"    → 上游来源: {row['Connections']}")

print("\n=== 流出 FF (outbound) ===")
outbound = ff_parsed_df[ff_parsed_df['Flow_Type'] == 'outbound']
for _, row in outbound.head(5).iterrows():
    print(f"  {row['ID']} 在 {row['Fwy']}{row['Dir']}: {row['Name']}")
    print(f"    → 下游目标: {row['Connections']}")

=== 流入 FF (inbound) ===
  311930 在 50E: 5NB and 5SB to 50EB
    → 上游来源: [('5', 'N'), ('5', 'S')]
  318276 在 50W: 99NB to 50WB FF
    → 上游来源: [('99', 'N')]
  318736 在 80E: 50EB to 80EB
    → 上游来源: [('50', 'E')]
  3026053 在 5N: 50EB to 5NB
    → 上游来源: [('50', 'E')]
  3070035 在 50E: 51SB TO 50EB
    → 上游来源: [('51', 'S')]

=== 流出 FF (outbound) ===
  312569 在 99N: 99NB to 50
    → 下游目标: []
  314943 在 50W: 50WB to 5NB
    → 下游目标: [('5', 'N')]
  3026111 在 160N: 160NB to 80EB FF
    → 下游目标: [('80', 'E')]
  3413035 在 5N: 5NB to 80EB
    → 下游目标: [('80', 'E')]
  3415025 在 99N: 99NB to 70EB
    → 下游目标: [('70', 'E')]


In [8]:
# 展开为每个 connection 一行
ff_expanded = []

for _, row in ff_parsed_df.iterrows():
    if len(row['Connections']) == 0:
        # 没有解析出连接，跳过
        ff_expanded.append({
            **row.to_dict(),
            'Connect_Fwy': None,
            'Connect_Dir': None,
        })
    else:
        for conn_fwy, conn_dir in row['Connections']:
            ff_expanded.append({
                **row.to_dict(),
                'Connect_Fwy': conn_fwy,
                'Connect_Dir': conn_dir,
            })

ff_expanded_df = pd.DataFrame(ff_expanded)
print(f"展开后记录数: {len(ff_expanded_df)}")
print(f"\n连接高速分布:")
print(ff_expanded_df['Connect_Fwy'].value_counts().head(10))
print(f"\n按流向统计:")
print(ff_expanded_df.groupby('Flow_Type')['ID'].count())

展开后记录数: 30

连接高速分布:
Connect_Fwy
80     8
50     4
99     3
5      3
51     3
65     2
70     2
505    1
Name: count, dtype: int64

按流向统计:
Flow_Type
inbound      8
outbound     6
unknown     16
Name: ID, dtype: int64


## 3. OSRM 路径查询

In [9]:
def get_osrm_route(src_lon, src_lat, dst_lon, dst_lat, server=OSRM_SERVER):
    """
    查询 OSRM 路径
    
    返回:
        distance: 距离（米）
        duration: 时间（秒）
        geometry: 路径坐标列表 [(lat, lon), ...]
    """
    url = f"{server}/route/v1/driving/{src_lon},{src_lat};{dst_lon},{dst_lat}"
    params = {
        'overview': 'full',
        'geometries': 'polyline'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if data.get('code') != 'Ok':
            return None, None, None
        
        route = data['routes'][0]
        distance = route['distance']
        duration = route['duration']
        
        # 解码 polyline
        geometry = polyline.decode(route['geometry'])
        
        return distance, duration, geometry
    except Exception as e:
        print(f"OSRM 查询失败: {e}")
        return None, None, None


# 测试 OSRM 连接
test_dist, test_dur, test_geom = get_osrm_route(-121.5, 38.5, -121.6, 38.6)
if test_dist:
    print(f"OSRM 连接正常: 测试距离 {test_dist:.0f}m")
else:
    print("警告: OSRM 连接失败，请检查服务器")

OSRM 连接正常: 测试距离 21020m


In [10]:
def find_connected_station(ff_row, ml_hv_df):
    """
    为 FF 找到连接的站点
    
    - 流入 (inbound): FF 是终点，搜索上游站点 → 路径方向: 上游 -> FF
    - 流出 (outbound): FF 是起点，搜索下游站点 → 路径方向: FF -> 下游
    """
    if pd.isna(ff_row['Connect_Fwy']) or pd.isna(ff_row['Connect_Dir']):
        return None
    
    # 筛选目标高速的同方向站点
    candidates = ml_hv_df[
        (ml_hv_df['Fwy'] == ff_row['Connect_Fwy']) &
        (ml_hv_df['Dir'] == ff_row['Connect_Dir'])
    ]
    
    if len(candidates) == 0:
        return None
    
    ff_lon, ff_lat = ff_row['Longitude'], ff_row['Latitude']
    flow_type = ff_row['Flow_Type']
    
    best_station = None
    best_distance = float('inf')
    best_geometry = None
    
    for _, cand in candidates.iterrows():
        if flow_type == 'inbound':
            # 流入: 从候选站点到 FF
            dist, dur, geom = get_osrm_route(
                cand['Longitude'], cand['Latitude'],
                ff_lon, ff_lat
            )
        else:
            # 流出: 从 FF 到候选站点
            dist, dur, geom = get_osrm_route(
                ff_lon, ff_lat,
                cand['Longitude'], cand['Latitude']
            )
        
        if dist is not None and dist < best_distance:
            best_distance = dist
            best_station = cand.to_dict()
            best_geometry = geom
    
    if best_station:
        return {
            'connected_id': best_station['ID'],
            'connected_fwy': best_station['Fwy'],
            'connected_dir': best_station['Dir'],
            'connected_type': best_station['Type'],
            'connected_abs_pm': best_station['Abs_PM'],
            'connected_lat': best_station['Latitude'],
            'connected_lon': best_station['Longitude'],
            'distance_m': best_distance,
            'geometry': best_geometry
        }
    
    return None


print("连接查找函数定义完成")

连接查找函数定义完成


In [11]:
# 为所有 FF 查找连接站点（这可能需要一些时间）
ff_connections = []

# 只处理有连接的 FF
ff_with_conn = ff_expanded_df[ff_expanded_df['Connect_Fwy'].notna()]
print(f"需要处理的 FF-连接对: {len(ff_with_conn)}")

for _, ff_row in tqdm(ff_with_conn.iterrows(), total=len(ff_with_conn), desc="查找连接"):
    connected = find_connected_station(ff_row, ml_hv_stations)
    
    connection = {
        'ff_id': ff_row['ID'],
        'ff_fwy': ff_row['Fwy'],
        'ff_dir': ff_row['Dir'],
        'ff_abs_pm': ff_row['Abs_PM'],
        'ff_lat': ff_row['Latitude'],
        'ff_lon': ff_row['Longitude'],
        'ff_name': ff_row['Name'],
        'flow_type': ff_row['Flow_Type'],
        'connect_fwy': ff_row['Connect_Fwy'],
        'connect_dir': ff_row['Connect_Dir'],
    }
    
    if connected:
        connection.update(connected)
        connection['status'] = 'found'
    else:
        connection['status'] = 'not_found'
    
    ff_connections.append(connection)

ff_conn_df = pd.DataFrame(ff_connections)
print(f"\n连接状态:")
print(ff_conn_df['status'].value_counts())
print(f"\n按流向统计:")
print(ff_conn_df[ff_conn_df['status'] == 'found'].groupby('flow_type')['ff_id'].count())

需要处理的 FF-连接对: 26


查找连接: 100%|██████████| 26/26 [00:14<00:00,  1.81it/s]


连接状态:
status
found    26
Name: count, dtype: int64

按流向统计:
flow_type
inbound      8
outbound     5
unknown     13
Name: ff_id, dtype: int64


In [12]:
# 查看找到的连接
found = ff_conn_df[ff_conn_df['status'] == 'found']
print(f"找到 {len(found)} 个连接")

print("\n=== 流入连接示例 (inbound) ===")
inbound = found[found['flow_type'] == 'inbound']
display_cols = ['ff_id', 'ff_fwy', 'ff_dir', 'ff_name', 'connected_id', 'connected_type', 'connected_fwy', 'connected_dir', 'distance_m']
print(inbound[display_cols].head(10).to_string(index=False))

print("\n=== 流出连接示例 (outbound) ===")
outbound = found[found['flow_type'] == 'outbound']
print(outbound[display_cols].head(10).to_string(index=False))

找到 26 个连接

=== 流入连接示例 (inbound) ===
  ff_id ff_fwy ff_dir             ff_name connected_id connected_type connected_fwy connected_dir  distance_m
 311930     50      E 5NB and 5SB to 50EB       314955             ML             5             N      4590.5
 311930     50      E 5NB and 5SB to 50EB       318484             ML             5             S      4794.0
 318276     50      W     99NB to 50WB FF       312566             ML            99             N      3369.7
 318736     80      E        50EB to 80EB       313824             ML            50             E      5250.9
3026053      5      N         50EB to 5NB      3090031             ML            50             E      1564.1
3070035     50      E        51SB TO 50EB      3070012             HV            51             S      4601.1
3413025      5      S         80WB to 5SB       320614             ML            80             W      5333.8
3415031     99      S        70WB to 99SB      3068052             ML            70 

In [13]:
# 保存连接结果
# 注意：geometry 列包含路径坐标，保存时需要转换
save_df = ff_conn_df.copy()
save_df['geometry'] = save_df['geometry'].apply(
    lambda x: str(x) if x is not None else None
)

output_file = os.path.join(OUTPUT_DIR, 'ff_connections.csv')
save_df.to_csv(output_file, index=False)
print(f"已保存: {output_file}")

已保存: ./output/ff_connection_analysis/ff_connections.csv


## 4. 可视化

In [14]:
def create_ff_connection_map(ff_conn_df, output_path, title="FF 连接器连边"):
    """
    创建 FF 连接器连边可视化地图
    
    显示:
    - FF 站点（黄色）
    - 连接的 ML/HV 站点（红/紫色）
    - OSRM 路径（流入蓝色，流出绿色）
    """
    found = ff_conn_df[ff_conn_df['status'] == 'found']
    
    if len(found) == 0:
        print("无有效连接")
        return None
    
    # 计算中心
    center_lat = found['ff_lat'].mean()
    center_lon = found['ff_lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles=None)
    
    # 底图
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr='Google', name='Google 街道'
    ).add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    # ========== OSRM 路径 ==========
    inbound_group = folium.FeatureGroup(name='流入路径 (inbound)')
    outbound_group = folium.FeatureGroup(name='流出路径 (outbound)')
    
    for _, row in found.iterrows():
        if row['geometry'] is None:
            continue
        
        geom = row['geometry']
        if isinstance(geom, str):
            continue
        
        flow_type = row['flow_type']
        if flow_type == 'inbound':
            color = '#2196F3'  # 蓝色
            popup_text = f"{row['connected_id']} → {row['ff_id']}<br>距离: {row['distance_m']:.0f}m"
            target_group = inbound_group
        else:
            color = '#4CAF50'  # 绿色
            popup_text = f"{row['ff_id']} → {row['connected_id']}<br>距离: {row['distance_m']:.0f}m"
            target_group = outbound_group
        
        folium.PolyLine(
            geom,
            color=color,
            weight=3,
            opacity=0.7,
            popup=popup_text
        ).add_to(target_group)
    
    inbound_group.add_to(m)
    outbound_group.add_to(m)
    
    # ========== FF 站点（黄色）==========
    ff_group = folium.FeatureGroup(name='FF 连接器')
    ff_shown = set()
    for _, row in found.iterrows():
        if row['ff_id'] in ff_shown:
            continue
        ff_shown.add(row['ff_id'])
        
        popup_text = f"""
        <b>FF 连接器</b><br>
        ID: {row['ff_id']}<br>
        Fwy: {row['ff_fwy']}{row['ff_dir']}<br>
        Abs_PM: {row['ff_abs_pm']:.2f}<br>
        Name: {row['ff_name']}<br>
        流向: {row['flow_type']}
        """
        
        folium.CircleMarker(
            [row['ff_lat'], row['ff_lon']],
            radius=10,
            color='#333',
            weight=2,
            fill=True,
            fillColor='#FFC107',
            fillOpacity=0.9,
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"FF {row['ff_id']}"
        ).add_to(ff_group)
    ff_group.add_to(m)
    
    # ========== 连接站点 ==========
    connected_group = folium.FeatureGroup(name='连接 ML/HV')
    type_colors = {'ML': '#E53935', 'HV': '#8E24AA'}
    connected_shown = set()
    
    for _, row in found.iterrows():
        if row['connected_id'] in connected_shown:
            continue
        connected_shown.add(row['connected_id'])
        
        color = type_colors.get(row['connected_type'], '#888')
        
        popup_text = f"""
        <b>连接站点</b><br>
        ID: {row['connected_id']}<br>
        Type: {row['connected_type']}<br>
        Fwy: {row['connected_fwy']}{row['connected_dir']}<br>
        Abs_PM: {row['connected_abs_pm']:.2f}
        """
        
        folium.CircleMarker(
            [row['connected_lat'], row['connected_lon']],
            radius=8,
            color='#333',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"{row['connected_type']} {row['connected_id']}"
        ).add_to(connected_group)
    connected_group.add_to(m)
    
    # 统计
    n_inbound = len(found[found['flow_type'] == 'inbound'])
    n_outbound = len(found[found['flow_type'] == 'outbound'])
    
    # 图例
    legend = f"""
    <div style="position:fixed; bottom:50px; left:50px; z-index:1000;
                background:white; padding:12px; border:2px solid #333; border-radius:5px;">
        <div style="font-weight:bold; margin-bottom:8px;">{title}</div>
        <div><span style="color:#FFC107;">●</span> FF 连接器</div>
        <div><span style="color:#E53935;">●</span> ML 主线</div>
        <div><span style="color:#8E24AA;">●</span> HV HOV</div>
        <div style="margin-top:5px;"><b>路径:</b></div>
        <div><span style="color:#2196F3;">—</span> 流入 (上游→FF): {n_inbound}</div>
        <div><span style="color:#4CAF50;">—</span> 流出 (FF→下游): {n_outbound}</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"已保存: {output_path}")
    return m


print("可视化函数定义完成")

可视化函数定义完成


In [15]:
# 生成全局地图
global_map = create_ff_connection_map(
    ff_conn_df,
    os.path.join(OUTPUT_DIR, 'ff_connections_all.html'),
    title="D3 FF 连接器连边"
)

if global_map:
    global_map

已保存: ./output/ff_connection_analysis/ff_connections_all.html


In [16]:
# 按 FF 所在高速分别生成地图
found = ff_conn_df[ff_conn_df['status'] == 'found']
ff_highways = found['ff_fwy'].unique()

for fwy in ff_highways:
    fwy_data = ff_conn_df[ff_conn_df['ff_fwy'] == fwy]
    if len(fwy_data[fwy_data['status'] == 'found']) > 0:
        create_ff_connection_map(
            fwy_data,
            os.path.join(OUTPUT_DIR, f'ff_connections_{fwy}.html'),
            title=f"Fwy {fwy} FF 连接器"
        )

已保存: ./output/ff_connection_analysis/ff_connections_50.html
已保存: ./output/ff_connection_analysis/ff_connections_51.html
已保存: ./output/ff_connection_analysis/ff_connections_5.html
已保存: ./output/ff_connection_analysis/ff_connections_80.html
已保存: ./output/ff_connection_analysis/ff_connections_65.html
已保存: ./output/ff_connection_analysis/ff_connections_160.html
已保存: ./output/ff_connection_analysis/ff_connections_99.html


## 5. 详细连接信息表

In [17]:
# 生成详细的连接信息表
found = ff_conn_df[ff_conn_df['status'] == 'found'].copy()

# 添加连接描述
def make_connection_desc(row):
    if row['flow_type'] == 'inbound':
        return f"{row['connected_fwy']}{row['connected_dir']} PM{row['connected_abs_pm']:.1f} → FF({row['ff_fwy']}{row['ff_dir']}) PM{row['ff_abs_pm']:.1f}"
    else:
        return f"FF({row['ff_fwy']}{row['ff_dir']}) PM{row['ff_abs_pm']:.1f} → {row['connected_fwy']}{row['connected_dir']} PM{row['connected_abs_pm']:.1f}"

found['connection'] = found.apply(make_connection_desc, axis=1)

# 选择需要的列
summary_cols = [
    'ff_id', 'ff_fwy', 'ff_dir', 'ff_abs_pm', 'ff_name', 'flow_type',
    'connected_id', 'connected_type', 'connected_fwy', 'connected_dir', 'connected_abs_pm',
    'distance_m', 'connection'
]

summary = found[summary_cols].copy()
summary['distance_m'] = summary['distance_m'].round(0).astype(int)

# 保存
summary_file = os.path.join(OUTPUT_DIR, 'ff_connections_summary.csv')
summary.to_csv(summary_file, index=False)
print(f"已保存: {summary_file}")

print(f"\n=== 流入连接 (inbound) ===")
print(summary[summary['flow_type'] == 'inbound'][['ff_id', 'ff_name', 'connected_id', 'connected_type', 'distance_m', 'connection']].head(10).to_string(index=False))

print(f"\n=== 流出连接 (outbound) ===")
print(summary[summary['flow_type'] == 'outbound'][['ff_id', 'ff_name', 'connected_id', 'connected_type', 'distance_m', 'connection']].head(10).to_string(index=False))

已保存: ./output/ff_connection_analysis/ff_connections_summary.csv

=== 流入连接 (inbound) ===
  ff_id             ff_name connected_id connected_type  distance_m                  connection
 311930 5NB and 5SB to 50EB       314955             ML        4590  5N PM518.5 → FF(50E) PM3.8
 311930 5NB and 5SB to 50EB       318484             ML        4794  5S PM518.4 → FF(50E) PM3.8
 318276     99NB to 50WB FF       312566             ML        3370 99N PM298.6 → FF(50W) PM5.3
 318736        50EB to 80EB       313824             ML        5251  50E PM0.6 → FF(80E) PM81.7
3026053         50EB to 5NB      3090031             ML        1564  50E PM2.9 → FF(5N) PM517.9
3070035        51SB TO 50EB      3070012             HV        4601   51S PM0.7 → FF(50E) PM5.6
3413025         80WB to 5SB       320614             ML        5334 80W PM86.8 → FF(5S) PM521.8
3415031        70WB to 99SB      3068052             ML        7647 70W PM3.5 → FF(99S) PM311.2

=== 流出连接 (outbound) ===
  ff_id          ff_nam

## 总结

### 流向判断逻辑

```
FF Name: "5NB to 50EB", FF 在 50E 上
→ 50E 在 "to" 后面 → 流入 (inbound)
→ 搜索 5N 作为上游
→ 路径方向: 5N 站点 → FF

FF Name: "80WB to 50WB", FF 在 80W 上
→ 80W 在 "to" 前面 → 流出 (outbound)
→ 搜索 50W 作为下游
→ 路径方向: FF → 50W 站点
```

### 输出文件

| 文件 | 说明 |
|------|------|
| `ff_connections.csv` | 完整连接结果 |
| `ff_connections_summary.csv` | 连接摘要 |
| `ff_connections_all.html` | 全局连接地图 |
| `ff_connections_{Fwy}.html` | 各高速单独地图 |

### 地图图例

| 颜色 | 含义 |
|------|------|
| 黄色 ● | FF 连接器 |
| 红色 ● | ML 主线站点 |
| 紫色 ● | HV HOV 站点 |
| 蓝色线 | 流入路径 (上游 → FF) |
| 绿色线 | 流出路径 (FF → 下游) |